In [45]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [57]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from SDRUtils.products.usd.usd_swaptions import USD_Swaptions
from SDRUtils.products._swaptions.pricer import (
    usd_swaption_straddle_pricer_from_row,
    usd_swaption_leg_pricer_from_row,
    usd_swaption_dealer_risk_reversal_skew_from_row,
    USDSwaptionStraddlePricerResult,
    USDSwaptionLegPricerResult,
	USDSwaptionDealerRiskReversalSkewResult,
    _compute_swaption_leg_greeks,
    USDSwaptionVerticalSpreadPricerResult,
    usd_swaption_vertical_spread_pricer_from_row
)

In [47]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"

as_of = datetime.date(2026, 1, 15)
start = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 0, 0))
end = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 23, 59))

mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
# df

MERGING SLICES...: 100%|██████████| 2/2 [00:00<00:00, 87.41it/s]


In [8]:
# sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path)
sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path, ignore_cache=True, merge_package_legs=True)
sdf

PRICING STRADDLES...: 100%|██████████| 226/226 [00:03<00:00, 63.06it/s]


,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,is_notional_capped,...,straddle_vega01,straddle_gamma01,straddle_theta1d,vega_curve_type,vega_curve_id,vega_curve_legs,vega_curve_vega01,vega_curve_weight,vega_curve_vega_ratio,vega_curve_pricing_method
0,MODI-TRAD,1732643977000000201,2026-01-15 05:13:03+00:00,2024-01-17 00:00:00,2026-01-20,SWAPTION_PAYER,USD-SOFR-COMPOUND 1D CONSTANT 2Yx1Y PAYER EURO...,120000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
1,MODI-TRAD,1732644545000000101,2026-01-15 05:13:34+00:00,2024-01-17 00:00:00,2026-01-20,SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 2Yx1Y PAYER ...,120000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
2,NEWT-NOVA,1743305405000000501,2026-01-15 10:31:26+00:00,2026-04-10 00:00:00,2026-04-10,SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 3Mx10Y PAYER...,18000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
3,NEWT-TRAD,1743318266000001901,2026-01-15 10:34:04+00:00,2027-03-15 00:00:00,2027-03-15,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT IMM_H2027xIM...,500000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
4,NEWT-TRAD,1743328875000000601,2026-01-15 10:36:32+00:00,2027-08-04 00:00:00,2027-08-04,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 1Y7Mx10Y REC...,25000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
310,NEWT-TRAD,1745369723000000101 / 1745369724000000201 / 17...,2026-01-15 19:31:22+00:00,2026-01-15 00:00:00,2026-07-15,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 6Mx1Y PAYER ...,500000000.0 / 200000000.0,USD,False,...,27539.005175,480.616987,-4307.966163,VEGA_DIAGONAL,VEGA_DIAGONAL_08644f1cdb15,"[1745369723000000101, 1745369724000000201, 174...",27539.005175174898 / 30242.377453203517,1.0,1.098165,QUANTLIB
311,NEWT-TRAD,1745385349000000401 / 1745385350000000501 / 17...,2026-01-15 19:34:01+00:00,2026-01-15 00:00:00,2026-07-15,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 6Mx1Y PAYER ...,250000000.0 / 70000000.0,USD,False,...,13769.502588,240.308493,-2153.983081,VEGA_TAIL_SPREAD,VEGA_TAIL_SPREAD_5733fb2fa5b7,"[1745385349000000401, 1745385350000000501, 174...",13769.502587587449 / 32439.000940767743,2.5,2.355859,QUANTLIB
312,NEWT-TRAD,1745415028000000301 / 1745415029000000401 / 17...,2026-01-15 19:38:27+00:00,2026-01-15 00:00:00,2027-01-15,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y PAYER...,50000000.0 / 70000000.0,USD,False,...,32281.925725,336.276395,-3151.324945,VEGA_EXPIRY_SPREAD,VEGA_EXPIRY_SPREAD_de64803e247a,"[1745415028000000301, 1745415029000000401, 174...",32281.925724813762 / 32439.000940767743,1.0,1.004866,QUANTLIB
313,NEWT-TRAD,1745610739000000601 / 1745610741000000801 / 17...,2026-01-15T20:47:06+00:00 / 2026-01-15T20:47:1...,2026-01-15 00:00:00,2026-03-16,SWAPTION_RECEIVER / SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT IMM_H2026xIM...,1000000000.0 / 50000000.0,USD,False,...,31880.838454,978.752962,-11724.421803,VEGA_DIAGONAL,VEGA_DIAGONAL_f255dd820209,"[1745610741000000801, 1745610739000000601, 174...",31880.838453769902 / 89031.290176405,3.0,2.792627,QUANTLIB


In [15]:
# sdf[(sdf["package_type"] == "STRADDLE") & (sdf["trade_label"].str.contains("3Mx7Y"))]["trade_label"].to_list()

# sdf["package_type"].value_counts()

# sdf[(sdf["trade_id"].str.contains("1744496707000000401"))]


# temp = sdf[(sdf["package_type"] == "STRADDLE") & (sdf["trade_label"].str.contains("1Yx10Y"))]
# temp
# ["platform_identifier"].value_counts()

# temp = sdf[sdf["package_type"].str.contains("SWAPTION")]
# temp["execution_timestamp"] = temp["execution_timestamp"].astype(str)
# temp.to_excel(f"{as_of.strftime("%Y-%m-%d")}_straddles.xlsx")
# temp
# ["trade_label"].value_counts()
# sdf[sdf["trade_label"].str.contains("1Yx10Y")]
# sdf.loc[361]
# ["trade_label"]

# sdf[sdf["package_type"].str.contains("RISK_R")]["platform_identifier"].value_counts()
sdf[sdf["package_type"].str.contains("VERT")]
# .to_dict(orient="records")
# ["platform_identifier"].value_counts()

# sdf[(sdf["package_type"].str.contains("RISK_R")) & (sdf["trade_label"].str.contains("2Yx10Y"))].to_dict(orient="records")
# temp["execution_timestamp"] = temp["execution_timestamp"].astype(str)
# temp.to_excel("2026-01-15_sdr_swaption.xlsx")

,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,is_notional_capped,...,straddle_vega01,straddle_gamma01,straddle_theta1d,vega_curve_type,vega_curve_id,vega_curve_legs,vega_curve_vega01,vega_curve_weight,vega_curve_vega_ratio,vega_curve_pricing_method
181,NEWT-TRAD,1741968482000002001 / 1741981511000002201,2026-01-15 13:03:30+00:00,2026-07-15 00:00:00,2026-07-15,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 6Mx30Y RECEI...,1e+08 / 2e+08,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
184,NEWT-TRAD,1742086978000000701 / 1742545921000000201,2026-01-15 13:19:28+00:00,2026-01-15 00:00:00,2027-01-15,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 1Yx1Y RECEIV...,100000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
188,TERM-NOVA,1742855448000001201 / 1742855449000001301,2026-01-15T13:50:02+00:00 / 2026-01-15T13:49:5...,2025-10-09 00:00:00,2026-04-09,SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 6Mx10Y PAYER...,18000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
189,TERM-NOVA,1742855443000000701 / 1742855445000000901,2026-01-15T13:50:09+00:00 / 2026-01-15T13:50:0...,2025-10-09 00:00:00,2026-04-09,SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 6Mx1Y PAYER ...,150000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
201,TERM-NOVA,1743301631000001901 / 1743311176000000101,2026-01-15T15:32:11+00:00 / 2026-01-15T15:33:1...,2026-04-10 00:00:00,2026-04-10,SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 6Mx10Y PAYER...,18000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
204,NEWT-NOVA,1743321046000001101 / 1743324035000000101,2026-01-15T15:35:01+00:00 / 2026-01-15T15:36:0...,2027-03-15 00:00:00,2027-03-15,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT IMM_H2027xIM...,5e+08 / 2.5e+08,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
215,TERM-EXER,1743489999000000101 / 1743491700000001301,2026-01-15T16:18:06+00:00 / 2026-01-15T16:19:4...,2025-10-15 00:00:00,2026-01-15,SWAPTION_PAYER,USD-SOFR-OIS Compound 1Y CONSTANT 3Mx1Y PAYER ...,1000000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
216,NEWT-TRAD,1743487258000000801 / 1743487261000001101,2026-01-15T16:18:08+00:00 / 2026-01-15T16:18:1...,2026-01-15 00:00:00,2026-10-22,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 9Mx10Y RECEI...,200000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
217,NEWT-TRAD,1743487256000000601 / 1743487259000000901,2026-01-15T16:18:08+00:00 / 2026-01-15T16:18:1...,2026-01-15 00:00:00,2029-01-16,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 3Yx10Y RECEI...,100000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
218,NEWT-TRAD,1743542012000000401 / 1743715554000000201,2026-01-15T16:18:52+00:00 / 2026-01-15T16:18:5...,2026-01-15 00:00:00,2027-01-15,SWAPTION_RECEIVER,USD-SOFR-COMPOUND 1D CONSTANT 1Yx3Y RECEIVER E...,300000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None


In [58]:
import ujson as json


def format_swaption_pricing_results(
    results: USDSwaptionStraddlePricerResult | USDSwaptionLegPricerResult | USDSwaptionDealerRiskReversalSkewResult,
):
    if isinstance(results, USDSwaptionDealerRiskReversalSkewResult):
        output = {
            "trade": results.trade_label,
            "atm_strike": results.atm_strike * 100,
            "otm_payer_strike": results.otm_payer_strike * 100,
            "otm_receiver_strike": results.otm_receiver_strike * 100,
            "wing_strike_width": results.wing_strike_width,
            "atm_bpvol": results.atm_bpvol_yr,
            "otm_payer_bpvol": results.otm_payer_bpvol_yr,
            "otm_receiver_bpvol": results.otm_receiver_bpvol_yr,
            "payer_skew_bpvol_yr": results.payer_skew_bpvol_yr,
            "receiver_skew_bpvol_yr": results.receiver_skew_bpvol_yr,
            "skew_bpvol": results.skew_bpvol_yr,
            "atm_notional": results.atm_notional,
            "wing_notional": results.wing_notional,
            "otm_payer_vega01": results.otm_payer_vega01,
            "otm_receiver_vega01": results.otm_receiver_vega01,
            "dv01": results.dv01,
            "gamma01": results.gamma01,
            "vega01": results.vega01,
            "theta1d": results.theta1d,
            "wing_dv01": results.wing_dv01
        }
    elif isinstance(results, USDSwaptionVerticalSpreadPricerResult):
        output = {
            "trade": results.trade_label,
            "spread_type": results.spread_type,
            # Strikes
            "atm_strike": results.atm_strike * 100,
            "otm_strike": results.otm_strike * 100,
            "strike_width_bps": results.strike_width_bps,
            "atm_strike_offset": results.atm_strike_offset,
            "otm_strike_offset": results.otm_strike_offset,
            # Vols
            "atm_bpvol": results.atm_bpvol_yr,
            "otm_bpvol": results.otm_bpvol_yr,
            "vol_spread_bpvol": results.vol_spread_bpvol_yr,
            # Notionals
            "atm_notional": results.atm_notional,
            "otm_notional": results.otm_notional,
            "notional_ratio": results.notional_ratio,
            # Premiums
            "net_premium": results.net_premium,
            "atm_premium": results.atm_premium,
            "otm_premium": results.otm_premium,
            # ATM leg Greeks
            "atm_dv01": results.atm_dv01,
            "atm_gamma01": results.atm_gamma01,
            "atm_vega01": results.atm_vega01,
            
            "atm_theta1d": results.atm_theta1d,
            # OTM leg Greeks
            "otm_dv01": results.otm_dv01,
            "otm_gamma01": results.otm_gamma01,
            "otm_vega01": results.otm_vega01,
            "otm_theta1d": results.otm_theta1d,
            # Aggregate Greeks
            "dv01": results.dv01,
            "gamma01": results.gamma01,
            "vega01": results.vega01,
            "theta1d": results.theta1d,
        }
    else:
        output = {
            "trade": results.trade_label,
            "prem": (results.fwd_prem / results.notional) * 10_000,
            "bpvol": results.bpvol_yr,
            "bpvol_day": results.bpvol_yr / np.sqrt(252),
            "dv01": results.dv01,
            "gamma01": results.gamma01,
            "vega01": results.vega01,
            "theta1d": results.theta1d,
        }

    print(json.dumps(output, indent=4))

In [60]:
format_swaption_pricing_results(usd_swaption_vertical_spread_pricer_from_row(sdf.loc[276], pricer))

{
    "trade": "USD-SOFR 1D CONSTANT 3Mx5Y PAYER EURO VANILLA PHYS",
    "spread_type": "PAYER_SPREAD",
    "atm_strike": 3.55,
    "otm_strike": 3.8,
    "strike_width_bps": 25.00000000000002,
    "atm_strike_offset": 0,
    "otm_strike_offset": 25,
    "atm_bpvol": 58.730684507544176,
    "otm_bpvol": 60.517480390056974,
    "vol_spread_bpvol": 1.7867958825127985,
    "atm_notional": 470000000.0,
    "otm_notional": 470000000.0,
    "notional_ratio": 1.0,
    "net_premium": 1604900.0,
    "atm_premium": 2231560.0,
    "otm_premium": 626660.0,
    "atm_dv01": 99063.64677939644,
    "atm_gamma01": 1066.5395679801943,
    "atm_vega01": 42405.765550686934,
    "atm_theta1d": 13874.553486622404,
    "otm_dv01": 38516.99807962427,
    "otm_gamma01": 1058.3048965203798,
    "otm_vega01": 28048.83789489761,
    "otm_theta1d": 9434.512999816448,
    "dv01": 60546.64869977217,
    "gamma01": 8.234671459814535,
    "vega01": 14356.927655789324,
    "theta1d": 4440.040486805956
}


In [18]:
# format_swaption_pricing_results(usd_swaption_straddle_pricer_from_row(sdf.loc[199], pricer))
# format_swaption_pricing_results(usd_swaption_leg_pricer_from_row(sdf.loc[96], pricer))

# format_swaption_pricing_results(usd_swaption_dealer_risk_reversal_skew_from_row(risk_reversal_row=sdf.loc[286], pricer=pricer))

In [53]:

row = sdf.loc[168]

display(row.to_dict())
_compute_swaption_leg_greeks(
	pricer,
	row["expiration_date"],
	row["underlying_expiration_date"],
	row["strike"],
	row["notional"],
	row["premium"],
	"receiver" if "rec" in row["product_type"].lower() else "payer",
)

{'event_action': 'NEWT-TRAD',
 'trade_id': '1745734985000000401',
 'execution_timestamp': Timestamp('2026-01-15 20:56:16+0000', tz='UTC'),
 'effective_date': Timestamp('2026-01-15 00:00:00'),
 'expiration_date': Timestamp('2028-01-10 00:00:00'),
 'product_type': 'SWAPTION_PAYER',
 'trade_label': 'USD-SOFR-OIS Compound 1Y CONSTANT 2Yx30Y PAYER EURO VANILLA PHYS',
 'notional': 100000000.0,
 'notional_currency': 'USD',
 'is_notional_capped': False,
 'package_type': 'SWAPTION',
 'package_id': None,
 'package_legs': None,
 'underlying_expiration_date': Timestamp('2058-01-12 00:00:00'),
 'tenor_years': 30.027397260273972,
 'tenor_label': '30Y',
 'forward_start_years': 1.9863013698630136,
 'forward_label': '2Y',
 'premium': 450000.0,
 'exercise_style': 'EUROPEAN',
 'strike': 0.05234,
 'upi_underlier_name': 'NA/Swap OIS USD',
 'unique_product_identifier': 'QZZLNQ2D4JQT',
 'platform_identifier': 'BILT',
 'cleared': 'N',
 'package_indicator': False,
 'package_transaction_price': '',
 'option_pre

_SwaptionLegGreeks(bpvol_yr=50.80008511489668, dv01=13146.336552531715, gamma01=394.24023436883004, vega01=34351.294088425864, theta1d=1203.1005319558317, strike_offset=100)

In [452]:
# temp = sdf.loc[422].copy()

# temp["premium"] = temp["premium"] / 4
# format_swaption_pricing_results(usd_swaption_leg_pricer_from_row(temp, pricer))
# format_swaption_pricing_results(usd_swaption_straddle_pricer_from_row(temp, pricer))

In [123]:
# ids = [1712299926000000501, 1712299925000000401]

ids = [
1745547082000000201,
1745547081000000101


]

# df[df["Dissemination Identifier"].isin([str(id) for id in ids])].to_csv("_temp_raw_raw_trades.csv",index=False)

df[df["Dissemination Identifier"].isin([str(id) for id in ids])].to_dict(orient="records")
# df[df["Original Dissemination Identifier"].isin([str(id) for id in ids])].to_dict(orient="records")
# df["Action type"].value_counts()
# df["Event type"].value_counts()

[{'Dissemination Identifier': '1745547082000000201',
  'Original Dissemination Identifier': '',
  'Action type': 'NEWT',
  'Event type': 'TRAD',
  'Event timestamp': Timestamp('2026-01-15 19:23:44+0000', tz='UTC'),
  'Amendment indicator': None,
  'Asset Class': 'IR',
  'Product name': None,
  'Cleared': 'N',
  'Mandatory clearing indicator': False,
  'Execution Timestamp': Timestamp('2026-01-15 19:23:44+0000', tz='UTC'),
  'Effective Date': Timestamp('2026-01-15 00:00:00'),
  'Expiration Date': Timestamp('2026-02-17 00:00:00'),
  'Maturity date of the underlier': datetime.date(2056, 2, 19),
  'Non-standardized term indicator': False,
  'Platform identifier': 'BILT',
  'Prime brokerage transaction indicator': False,
  'Block trade election indicator': False,
  'Large notional off-facility swap election indicator': False,
  'Notional amount-Leg 1': '60,000,000',
  'Notional amount-Leg 2': '60,000,000',
  'Notional currency-Leg 1': 'USD',
  'Notional currency-Leg 2': 'USD',
  'Notional q